In [10]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ultralytics>=8.3.100", "fiftyone"])
print("Done. Restart runtime, then run from Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.9/935.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/20

In [11]:
import os, subprocess, sys

# Uses the Safety Helmet Detection dataset from Kaggle Datasets directly.
# No API key needed inside a Kaggle notebook — kaggle API is pre-authenticated.
DATASET_DIR = "/kaggle/working/helmet_data"
os.makedirs(DATASET_DIR, exist_ok=True)

subprocess.check_call([
    "kaggle", "datasets", "download",
    "-d", "andrewmvd/hard-hat-detection",
    "--unzip", "-p", DATASET_DIR
])

# Verify what we got
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files[:3]:
        print("  " * (level + 1) + f)
    if files and len(files) > 3:
        print("  " * (level + 1) + f"... ({len(files)} files total)")

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection
License(s): CC0-1.0


100%|██████████| 1.22G/1.22G [00:07<00:00, 182MB/s] 



helmet_data/
  images/
    hard_hat_workers649.png
    hard_hat_workers3135.png
    hard_hat_workers11.png
    ... (5000 files total)
  annotations/
    hard_hat_workers1810.xml
    hard_hat_workers507.xml
    hard_hat_workers4953.xml
    ... (5000 files total)


In [12]:
import xml.etree.ElementTree as ET
import shutil, os
from pathlib import Path

SRC_IMG = os.path.join(DATASET_DIR, "images")
SRC_ANN = os.path.join(DATASET_DIR, "annotations")
OUT_DIR = "/kaggle/working/helmet_yolo"

# Classes that map to our two pipeline labels
# Pipeline check_helmet() looks for "helmet" and "no_helmet" in class name
CLASS_MAP = {
    "helmet":    0,   # has helmet
    "head":      1,   # no helmet (bare head)
    "person":    -1,  # skip — we only care about head-level crops
}
CLASS_NAMES = ["helmet", "no_helmet"]  # idx 0 and 1

for split in ["train", "val"]:
    os.makedirs(f"{OUT_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{OUT_DIR}/{split}/labels", exist_ok=True)

img_files = sorted(Path(SRC_IMG).glob("*.png")) + sorted(Path(SRC_IMG).glob("*.jpg"))
print(f"Total images found: {len(img_files)}")

# 90/10 split
split_idx = int(len(img_files) * 0.9)
splits = {"train": img_files[:split_idx], "val": img_files[split_idx:]}

skipped = 0
for split_name, files in splits.items():
    for img_path in files:
        ann_path = Path(SRC_ANN) / (img_path.stem + ".xml")
        if not ann_path.exists():
            skipped += 1
            continue

        tree = ET.parse(ann_path)
        root = tree.getroot()
        size = root.find("size")
        W = int(size.find("width").text)
        H = int(size.find("height").text)

        lines = []
        for obj in root.findall("object"):
            name = obj.find("name").text.lower().strip()
            if name not in CLASS_MAP or CLASS_MAP[name] == -1:
                continue
            cls_idx = CLASS_MAP[name]
            bb = obj.find("bndbox")
            x1 = int(bb.find("xmin").text)
            y1 = int(bb.find("ymin").text)
            x2 = int(bb.find("xmax").text)
            y2 = int(bb.find("ymax").text)
            cx = ((x1 + x2) / 2) / W
            cy = ((y1 + y2) / 2) / H
            bw = (x2 - x1) / W
            bh = (y2 - y1) / H
            lines.append(f"{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if not lines:
            skipped += 1
            continue

        shutil.copy(img_path, f"{OUT_DIR}/{split_name}/images/{img_path.name}")
        with open(f"{OUT_DIR}/{split_name}/labels/{img_path.stem}.txt", "w") as f:
            f.write("\n".join(lines))

print(f"Conversion done. Skipped: {skipped}")
print(f"  Train images: {len(list(Path(OUT_DIR+'/train/images').glob('*')))}")
print(f"  Val   images: {len(list(Path(OUT_DIR+'/val/images').glob('*')))}")

Total images found: 5000
Conversion done. Skipped: 0
  Train images: 4500
  Val   images: 500


In [13]:
import yaml

DATASET_YAML = f"{OUT_DIR}/data.yaml"
data = {
    "train": f"{OUT_DIR}/train/images",
    "val":   f"{OUT_DIR}/val/images",
    "nc":    2,
    "names": ["helmet", "no_helmet"],   # matches pipeline check_helmet() logic
}
with open(DATASET_YAML, "w") as f:
    yaml.dump(data, f)

print(f"data.yaml written: {DATASET_YAML}")
print(data)

data.yaml written: /kaggle/working/helmet_yolo/data.yaml
{'train': '/kaggle/working/helmet_yolo/train/images', 'val': '/kaggle/working/helmet_yolo/val/images', 'nc': 2, 'names': ['helmet', 'no_helmet']}


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data     = DATASET_YAML,
    epochs   = 50,
    imgsz    = 416,      # head crops are small — 416 is plenty, faster than 640
    batch    = 32,
    device   = 0,
    project  = "/kaggle/working/helmet_training",
    name     = "helmet_v1",
    patience = 20,
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    degrees  = 10.0,
    fliplr   = 0.5,
    mosaic   = 1.0,
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/helmet_yolo/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_v12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

In [ ]:
import shutil, os
from ultralytics import YOLO

best_weights = os.path.join(str(results.save_dir), "weights", "best.pt")
output_path  = "/kaggle/working/helmet_detector.pt"
shutil.copy(best_weights, output_path)

print(f"Saved: {output_path}")
print(f"mAP50: {results.results_dict.get('metrics/mAP50(B)', 'see results')}")

model_check = YOLO(output_path)
print(f"Classes: {model_check.names}")
# Must contain 'helmet' and 'no_helmet' — pipeline depends on these names
assert any("helmet" in n for n in model_check.names.values()), "Missing helmet class!"
assert any("no_helmet" in n for n in model_check.names.values()), "Missing no_helmet class!"
print("Verification passed — upload helmet_detector.pt to cbms-helmet-model dataset.")

---
## Helmet API — Cell A : Server Dependencies
Install server dependencies (ngrok, fastapi, uvicorn). Run once, then restart session if needed.

In [ ]:
!pip install -q httpx pyngrok nest-asyncio fastapi uvicorn python-multipart
print("Server deps ready.")

## Helmet API — Cell C : Configuration & Imports

In [ ]:
import os, cv2, json, base64, time, shutil, re, threading
import numpy as np
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, FileResponse
import uvicorn
from ultralytics import YOLO

try:
    from kaggle_secrets import UserSecretsClient
    NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = ""   # paste your token here if running outside Kaggle

# ── Paths ────────────────────────────────────────────────────────────────
HELMET_MODEL_PATH = "/kaggle/working/helmet_detector.pt"   # produced by training cell

# ── Detection thresholds ─────────────────────────────────────────────────
HELMET_CONF        = 0.40   # YOLO confidence threshold
ALERT_NO_HELMET    = True   # generate alert when bare head detected
NO_HELMET_CLASS    = "no_helmet"  # must match training class name
JPEG_QUALITY       = 70
EVIDENCE_PRE       = 3
EVIDENCE_POST      = 7
EVIDENCE_TOTAL     = EVIDENCE_PRE + EVIDENCE_POST

# Confirm model exists before starting server
if not os.path.exists(HELMET_MODEL_PATH):
    print(f"[WARN] Model not found at {HELMET_MODEL_PATH}.")
    print("  Run the training cells above first, then re-run this cell.")
else:
    print(f"[OK] Helmet model found: {HELMET_MODEL_PATH}")

print("[DEBUG] Helmet API config loaded.")

## Helmet API — Cell D : Load Helmet Model

In [ ]:
import torch

print("Loading helmet detection model...")
HELMET_MODEL = YOLO(HELMET_MODEL_PATH)
if torch.cuda.is_available():
    HELMET_MODEL.to("cuda")
    print("  Helmet YOLO -> GPU")
else:
    print("  Helmet YOLO -> CPU")

print(f"  Classes: {HELMET_MODEL.names}")
assert any("no_helmet" in n for n in HELMET_MODEL.names.values()), \
    "Model missing 'no_helmet' class — did training complete?"
print("[DEBUG] Helmet model ready.")

## Helmet API — Cell E : StatefulHelmetPipeline

In [ ]:
from collections import defaultdict, deque


class StatefulHelmetPipeline:
    """
    Maintains per-track state across sequential video chunks.
    Detects helmets/bare heads using the trained YOLOv8 model,
    and generates an alert whenever a bare head (no_helmet) is
    observed for CONFIRMATION_FRAMES consecutive frames.
    """

    CONFIRMATION_FRAMES = 5   # consecutive no_helmet detections before alert

    def __init__(self):
        self.frame_idx     = 0
        self.pre_buffers   = defaultdict(lambda: deque(maxlen=EVIDENCE_PRE))
        self.capture_state = {}
        self.person_scores = defaultdict(lambda: 100)
        # count consecutive no_helmet frames per track
        self.no_helmet_streak = defaultdict(int)
        self.track_last_seen  = {}

    # ------------------------------------------------------------------
    def _make_evidence_grid(self, crops):
        """Encode a contact-sheet of crop thumbnails as base-64 JPEG."""
        blank  = np.zeros((128, 128, 3), dtype=np.uint8)
        thumbs = [
            cv2.resize(c, (128, 128)) if c is not None and c.size > 0 else blank
            for c in crops
        ]
        while len(thumbs) < EVIDENCE_TOTAL:
            thumbs.append(blank)
        grid = np.vstack([np.hstack(thumbs[:5]), np.hstack(thumbs[5:10])])
        _, buf = cv2.imencode(".jpg", grid, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        return base64.b64encode(buf).decode()

    # ------------------------------------------------------------------
    def process_chunk(self, video_path: str):
        cap    = cv2.VideoCapture(video_path)
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps    = cap.get(cv2.CAP_PROP_FPS) or 15.0

        output_path = "/kaggle/working/helmet_processed_" + os.path.basename(video_path)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        alerts_generated = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            h_f, w_f = frame.shape[:2]

            try:
                results = HELMET_MODEL.track(
                    frame,
                    conf=HELMET_CONF,
                    persist=True,
                    verbose=False,
                )[0]
            except Exception as e:
                print(f"[ERROR] YOLO tracking failed: {e}")
                out.write(frame)
                self.frame_idx += 1
                continue

            active_tids = set()

            if results.boxes is not None and results.boxes.id is not None:
                for box in results.boxes:
                    try:
                        tid       = int(box.id[0])
                        cls_id    = int(box.cls[0])
                        cls_name  = HELMET_MODEL.names[cls_id]
                        conf      = float(box.conf[0])

                        active_tids.add(tid)
                        self.track_last_seen[tid] = self.frame_idx

                        x1 = max(0,   int(box.xyxy[0][0]))
                        y1 = max(0,   int(box.xyxy[0][1]))
                        x2 = min(w_f, int(box.xyxy[0][2]))
                        y2 = min(h_f, int(box.xyxy[0][3]))
                        if x2 <= x1 or y2 <= y1:
                            continue

                        crop = frame[y1:y2, x1:x2].copy()
                        self.pre_buffers[tid].append(crop)

                        is_violation = (cls_name == NO_HELMET_CLASS)

                        if is_violation:
                            self.no_helmet_streak[tid] += 1
                        else:
                            self.no_helmet_streak[tid] = 0

                        # Trigger alert after CONFIRMATION_FRAMES consecutive violations
                        if (ALERT_NO_HELMET
                                and is_violation
                                and self.no_helmet_streak[tid] == self.CONFIRMATION_FRAMES
                                and tid not in self.capture_state):
                            self.capture_state[tid] = {
                                "frames": list(self.pre_buffers[tid]),
                                "conf":   conf,
                            }
                            print(f"[DEBUG] Capture triggered for TID {tid}: no_helmet")

                        # Finalise alert once enough post-event frames collected
                        if tid in self.capture_state:
                            cs = self.capture_state[tid]
                            cs["frames"].append(crop)
                            if len(cs["frames"]) >= EVIDENCE_TOTAL:
                                grid  = self._make_evidence_grid(cs["frames"])
                                alert = {
                                    "person_name":       f"UNKNOWN-{tid}",
                                    "identity":          "unknown",
                                    "activity":          "no_helmet",
                                    "activity_conf":     round(cs["conf"], 3),
                                    "id_confidence":     0.0,
                                    "new_score":         0,
                                    "evidence_grid_b64": grid,
                                }
                                alerts_generated.append(alert)
                                print(f"[ALERT] No-helmet alert for TID {tid}")
                                del self.capture_state[tid]
                                self.no_helmet_streak[tid] = 0

                        # Draw bounding box
                        color = (0, 0, 220) if is_violation else (0, 200, 80)
                        label = f"ID:{tid} {cls_name} {conf:.0%}"
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        cv2.putText(frame, label,
                                    (x1, max(y1 - 8, 12)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

                    except Exception as e:
                        print(f"[ERROR] Box processing error TID: {e}")

            # Prune stale tracks
            stale = [t for t, last in self.track_last_seen.items()
                     if (self.frame_idx - last) > 60]
            for t in stale:
                del self.track_last_seen[t]
                self.no_helmet_streak.pop(t, None)

            self.frame_idx += 1
            out.write(frame)

        cap.release()
        out.release()
        return alerts_generated, output_path


HELMET_PIPELINE = StatefulHelmetPipeline()
print("[DEBUG] StatefulHelmetPipeline instantiated.")

## Helmet API — Cell F : FastAPI Endpoints

In [ ]:
import asyncio

helmet_app  = FastAPI(title="CBMS Helmet Detection API")

# Sequential processing lock (same pattern as cbms-pipeline)
_hlock            = threading.Lock()
_hcond            = threading.Condition(_hlock)
_expected_chunk   = 0


@helmet_app.post("/process_chunk")
async def process_chunk(file: UploadFile = File(...)):
    """Upload a .mp4 chunk; returns annotated video + alerts in headers.

    Response headers
    ----------------
    X-Global-Frame  : running frame counter across all chunks
    X-Alerts        : JSON list of alert dicts (evidence grid stripped)
    X-Chunk-Idx     : echo of the submitted chunk index
    X-Pipeline      : always 'helmet'
    """
    global _expected_chunk

    m         = re.search(r'chunk_(\d+)', file.filename)
    chunk_idx = int(m.group(1)) if m else -1

    temp_path = f"/kaggle/working/helmet_temp_{file.filename}"
    with open(temp_path, "wb") as buf:
        shutil.copyfileobj(file.file, buf)

    try:
        with _hcond:
            if chunk_idx != -1:
                while chunk_idx != _expected_chunk:
                    _hcond.wait(timeout=1.0)

            print(f"[HELMET] Processing {file.filename} "
                  f"(idx={chunk_idx}, expected={_expected_chunk})")
            t0 = time.time()
            alerts, output_path = HELMET_PIPELINE.process_chunk(temp_path)
            print(f"[HELMET] Done in {time.time() - t0:.2f}s")

            if chunk_idx != -1:
                _expected_chunk = chunk_idx + 1
                _hcond.notify_all()

        # Strip evidence grid from header payload (too large)
        slim_alerts = []
        for a in alerts:
            ac = dict(a)
            ac.pop("evidence_grid_b64", None)
            slim_alerts.append(ac)

        headers = {
            "X-Global-Frame": str(HELMET_PIPELINE.frame_idx),
            "X-Alerts":       json.dumps(slim_alerts),
            "X-Chunk-Idx":    str(chunk_idx),
            "X-Pipeline":     "helmet",
        }
        return FileResponse(path=output_path, headers=headers,
                            media_type="video/mp4")

    except Exception as e:
        print(f"[ERROR] process_chunk failed: {e}")
        return JSONResponse({"error": str(e)}, status_code=500)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


@helmet_app.get("/scores")
def get_scores():
    """GET /scores — per-person cumulative behaviour scores."""
    return JSONResponse(content=dict(HELMET_PIPELINE.person_scores))


@helmet_app.get("/health")
def health():
    """GET /health — liveness probe."""
    return {
        "status":         "ok",
        "pipeline":       "helmet",
        "global_frame":   HELMET_PIPELINE.frame_idx,
        "enrolled":       0,
        "expected_chunk": _expected_chunk,
    }


@helmet_app.post("/reset_pipeline")
def reset_pipeline():
    """POST /reset_pipeline — re-create the pipeline and reset chunk counter."""
    global HELMET_PIPELINE, _expected_chunk
    with _hcond:
        HELMET_PIPELINE  = StatefulHelmetPipeline()
        _expected_chunk  = 0
        _hcond.notify_all()
    print("[HELMET RESET] Pipeline and chunk counter reset.")
    return {"status": "reset"}


@helmet_app.post("/set_expected_chunk/{idx}")
def set_expected_chunk(idx: int):
    """POST /set_expected_chunk/{idx} — resync server chunk counter."""
    global _expected_chunk
    with _hcond:
        print(f"[HELMET RESYNC] Expecting chunk_{idx:04d} next.")
        _expected_chunk = idx
        _hcond.notify_all()
    return {"status": "resynced", "expected": _expected_chunk}


print("[DEBUG] Helmet FastAPI routes defined.")
print("  POST /process_chunk")
print("  GET  /scores")
print("  GET  /health")
print("  POST /reset_pipeline")
print("  POST /set_expected_chunk/{idx}")

## Helmet API — Cell G : Launch ngrok + Uvicorn
Run this cell last. Copy the URL printed below and paste it into `kaggle_client.py` → `HELMET_NGROK_URL`.

In [ ]:
import asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
tunnel     = ngrok.connect(8001)   # port 8001 — distinct from main pipeline (8000)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print(f"HELMET API LIVE at : {public_url}")
print(f"Send chunks to     : {public_url}/process_chunk")
print(f"Health check       : {public_url}/health")
print(f"Scores             : {public_url}/scores")
print("=" * 60)
print("\nPaste the URL above into kaggle_client.py -> HELMET_NGROK_URL")
print("Server running... interrupt kernel to stop.")

config = uvicorn.Config(
    helmet_app,
    host      = "0.0.0.0",
    port      = 8001,
    loop      = "asyncio",
    log_level = "warning",
)
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())